# 04 — Feature Engineering & ML Problem Selection
This notebook does two things:
1. Builds the causal (leak-free) feature set used for modeling
2. Lays out and compares the candidate ML/analytics problems, with a final recommendation
   backed by evidence from notebooks 01-03

In [1]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
from features.build_features import build_feature_set

df = pd.read_csv('../data/processed/jabalpur_clean.csv')
df['valid_time'] = pd.to_datetime(df['valid_time'])
df.shape

(17544, 24)

## Candidate ML/Analytics Problems

### A. Short-term temperature forecasting (regression)
- **Target:** `t2m_c`, 1-24h ahead
- **Inputs:** lagged temperature/pressure/wind/dewpoint, rolling stats, cyclical time-of-day/year features
- **Horizon:** 1h (primary), extensible to 24h
- **Why it matters:** temperature drives heat-stress warnings, agriculture, energy demand
- **Evidence:** lag-1 autocorrelation of 0.975, zero missing data, clean continuous target
- **Difficulty:** Low-Medium
- **Metrics:** MAE, RMSE, R²
- **SIH relevance:** Medium-High — a working, evidence-backed forecasting demo
- **Limitations:** single grid point, so not a spatial forecast; reanalysis data has some smoothing vs true point sensors

### B. Rain / no-rain occurrence classification (binary)
- **Target:** `rain_flag` (tp_mm > 0.1mm), hourly or daily
- **Inputs:** pressure trend, humidity, wind, lagged rain occurrence, calendar features
- **Horizon:** next 1-24h
- **Why it matters:** directly actionable for agriculture/disaster-prep — closest to real SIH "weather analytics" ask
- **Evidence:** 16% hourly / 31% daily positive rate — imbalanced but workable
- **Difficulty:** Medium
- **Metrics:** Precision, Recall, F1, ROC-AUC; must report class balance
- **SIH relevance:** High
- **Limitations:** still single-point; imbalance needs explicit handling (class weights / resampling)

### C. Heavy rainfall event detection (extreme classification)
- **Target:** daily rainfall > IMD "heavy rain" threshold (64.5mm/day)
- **Evidence:** only **2 positive days out of 731** in this dataset
- **Difficulty:** High — too few positives for a trustworthy baseline
- **Verdict:** not viable as the Phase-1 baseline on this dataset alone; revisit once IMD/MOSDAC/multi-station data is fused in (more locations = more positive events)

### D. Pressure-anomaly-based rain-onset signal
- Pressure anti-correlates with rainfall (-0.26) and temperature (-0.67), consistent with meteorology
- Useful as a **feature** for problems A/B, not yet a standalone target with this single-point dataset

## Recommendation
**Primary Phase-1 baseline: (A) short-term temperature forecasting (regression).**
Strongest, cleanest signal in the data, zero data-quality caveats, gives a clear MAE/RMSE/R² story.

**Secondary/parallel: (B) rain occurrence classification.** Higher SIH relevance; workable class balance;
built alongside as evidence that the pipeline generalizes beyond a single regression target.

**(C) is explicitly deferred to Phase 2**, once real IMD/MOSDAC/multi-station fusion increases the number
of observed extreme events — this keeps the SIH narrative honest and consistent with the original
fragmentation-problem framing.

## Build feature set for Problem A (temperature, 1h horizon)

In [2]:
feat_temp = build_feature_set(df, horizon=1)
print(feat_temp.shape)
feat_temp.filter(regex='t2m_c|target|hour_sin|hour_cos').head()

(17519, 71)


,t2m_c,hour_sin,hour_cos,t2m_c_lag1,t2m_c_lag2,t2m_c_lag3,t2m_c_lag6,t2m_c_lag12,t2m_c_lag24,t2m_c_rollmean3,t2m_c_rollstd3,t2m_c_rollmean6,t2m_c_rollstd6,t2m_c_rollmean24,t2m_c_rollstd24,target_t2m_h1
0,14.31210,0.000000,1.000000,13.29666,13.58688,14.69457,16.06450,22.68190,13.05460,13.859370,0.737717,14.835493,1.216132,18.218468,4.187720,14.33570
1,14.33570,0.258819,0.965926,14.31210,13.29666,13.58688,16.18197,22.27020,12.88530,13.731880,0.523018,14.543427,1.062690,18.270863,4.127739,14.86830
2,14.86830,0.500000,0.866025,14.33570,14.31210,13.29666,15.18838,20.43383,13.67098,13.981487,0.593195,14.235715,0.698106,18.331297,4.055446,14.85238
3,14.85238,0.707107,0.707107,14.86830,14.33570,14.31210,14.69457,18.20864,14.32992,14.505367,0.314531,14.182368,0.618430,18.381185,4.002645,15.69607
4,15.69607,0.866025,0.500000,14.85238,14.86830,14.33570,13.58688,18.30150,15.12643,14.685460,0.303006,14.208670,0.647255,18.402954,3.981016,19.35170


## Build feature set for Problem B (rain occurrence, 1h horizon)

In [3]:
from features.build_features import add_cyclical_time_features, add_lag_features, add_rolling_features

feat_rain = add_cyclical_time_features(df)
feat_rain = add_lag_features(feat_rain, cols=('t2m_c','msl_hpa','wind_speed','d2m_c','tp_mm'))
feat_rain = add_rolling_features(feat_rain, cols=('t2m_c','msl_hpa','tp_mm'))
feat_rain['target_rain_next1h'] = feat_rain['rain_flag'].shift(-1)
feat_rain = feat_rain.dropna().reset_index(drop=True)
print(feat_rain.shape)
print('Positive rate:', feat_rain['target_rain_next1h'].mean().round(3))

(17519, 77)
Positive rate: 0.162


In [4]:
import os
os.makedirs('../data/processed', exist_ok=True)
feat_temp.to_csv('../data/processed/features_temperature.csv', index=False)
feat_rain.to_csv('../data/processed/features_rain.csv', index=False)
print('Saved both feature sets.')

Saved both feature sets.


## Summary of Notebook 04
- Feature sets built causally (only past values, `.shift()`-based) — safe for chronological splitting, no leakage
- Problem A (temperature regression) feature set: lags, rolling stats, cyclical time features
- Problem B (rain classification) feature set: same style, target = next-hour rain occurrence
- Both saved to `data/processed/` for notebook 05
- Next: `05_baseline_model.ipynb`